### html + htmlheadertextsplitter+ remove html junk + better chunking

### all-MiniLM-L6-v2 + larger chunk size + excluding sources

In [2]:
import sys
import os
import chromadb
path = "C:\\Users\\luyer\\my_rag\\data"

In [3]:
client = chromadb.PersistentClient(path=path)

In [4]:
cosine_collection = client.get_or_create_collection(
    name="my_movies_html_config_cosine",
    configuration={
        "hnsw": {"space": "cosine",
                 "ef_construction": 200,
                 "ef_search": 200
                 }
    }
)

In [26]:
cosine_collection_large_chunk = client.get_or_create_collection(
    name="my_movies_html_config_cosine_large_chunk_no_resources",
    configuration={
        "hnsw": {"space": "cosine"
                 }
    }
)

In [3]:
collection_persistent = client.get_or_create_collection(
    name="my_movies_html_v3"
)

In [5]:
files = [
    file_name
    for file_name in os.listdir(path)
    if os.path.isfile(os.path.join(path, file_name)) and file_name.endswith('.html')
]
print(files)

['2001_A_Space_Odyssey.html', 'Casablanca.html', 'Citizen_Kane.html', 'Parasite_2019.html', 'Pulp_Fiction.html', 'Seven_Samurai.html', 'Spirited_Away.html', 'The_Dark_Knight.html', 'The_Godfather.html', 'The_Matrix.html']


In [20]:
import re
from bs4 import BeautifulSoup
from pathlib import Path
from langchain_text_splitters import (
    HTMLHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

def clean_chunk_text(text: str) -> str:
    """Clean text after HTML parsing."""
    text = text.replace("\x00", "")

    # remove Wikipedia citation markers like [1], [22], [123]
    text = re.sub(r"\[\d+\]", "", text)

    # normalize whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()

header_splitter = HTMLHeaderTextSplitter(
    headers_to_split_on=[
        ("h1", "header1"),
        ("h2", "header2"),
        ("h3", "header3"),
    ]
)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
    separators=[
        "\n\n",
        "\n",
        ". ",
        "? ",
        "! ",
        " ",
        "",
    ]
)

BAD_SECTIONS = {
    "References",
    "External links",
    "Further reading",
    "Sources"
}

all_chunks = []

for file_name in files:
    file_path = Path(path) / file_name

    raw_html = file_path.read_text(encoding="utf-8", errors="ignore")

    # ----------------------------
    # 1. Parse HTML
    # ----------------------------

    soup = BeautifulSoup(raw_html, "html.parser")

    # remove useless HTML elements
    for tag in soup([
        "script",
        "style",
        "nav",
        "footer",
        "noscript",
    ]):
        tag.decompose()

    # ----------------------------
    # 2. Keep article body only
    # ----------------------------

    content = soup.select_one("#mw-content-text")

    if content is None:
        content = soup.body

    if content is None:
        continue

    clean_html = str(content)

    # ----------------------------
    # 3. Split by HTML headings
    # ----------------------------

    section_chunks = header_splitter.split_text(clean_html)
    
    # ----------------------------
    # 4. Exclude bad sections
    # ----------------------------

    section_chunks = [
        doc
        for doc in section_chunks
        if doc.metadata.get("header2") not in BAD_SECTIONS
    ]

    chunks = text_splitter.split_documents(section_chunks)
    # ----------------------------
    # 6. Clean + enrich chunks
    # ----------------------------

    for chunk_index, chunk in enumerate(chunks):

        text = clean_chunk_text(chunk.page_content)

        if not text:
            continue

        h1 = chunk.metadata.get("header1", "")
        h2 = chunk.metadata.get("header2", "")
        h3 = chunk.metadata.get("header3", "")

        # Add structural context into embedded text
        prefix_parts = []

        if h1:
            prefix_parts.append(f"Article: {h1}")

        if h2:
            prefix_parts.append(f"Section: {h2}")

        if h3:
            prefix_parts.append(f"Subsection: {h3}")

        if prefix_parts:
            text = "\n".join(prefix_parts) + "\n\n" + text

        all_chunks.append(
            {
                "id": f"html-{file_name}-{chunk_index}",
                "document": text,
                "metadata": {
                    **chunk.metadata,
                    "source": file_name,
                },
            }
        )




In [7]:
cosine_collection.add(
    ids=[item["id"] for item in all_chunks],
    documents=[item["document"] for item in all_chunks],
    metadatas=[item["metadata"] for item in all_chunks],
)

print(f"Added {len(all_chunks)} HTML chunks")

Added 2327 HTML chunks


In [27]:
cosine_collection_large_chunk.add(
    ids=[item["id"] for item in all_chunks],
    documents=[item["document"] for item in all_chunks],
    metadatas=[item["metadata"] for item in all_chunks],
)

print(f"Added {len(all_chunks)} HTML chunks")

Added 1209 HTML chunks


In [8]:
import pprint

In [23]:
pprint.pprint(chunk.page_content)

('Babenko, Yelyzaveta (2011). . GRIN Verlag. .  \n'
 'Analysis of the Film the Matrix  \n'
 'ISBN  \n'
 '978-3-640-91285-8  \n'
 'Clover, Joshua (2004). . BFI. .  \n'
 'The Matrix  \n'
 'ISBN  \n'
 '978-1-84457-045-4  \n'
 'Condon, Paul (2003). . London: . .  \n'
 'The Matrix Unlocked: An Unauthorized Review of the Matrix Phenomenon  \n'
 'Contender Books  \n'
 'ISBN  \n'
 '978-1-84357-093-6  \n'
 'Irwin, William (2002). . Open Court. .  \n'
 'The Matrix and Philosophy: Welcome to the Desert of the Real  \n'
 'ISBN  \n'
 '978-0-8126-9502-1  \n'
 'Jones, Steven E. (2006). . Routledge. .  \n'
 'Against Technology: From the Luddites to Neo-Luddism  \n'
 'ISBN  \n'
 '978-0-415-97868-2  \n'
 'Pegg, Simon (2010). . Century. .  \n'
 'Nerd Do Well  \n'
 'ISBN  \n'
 '978-1-84605-811-0  \n'
 'Toropov, Brandon; Hansen, Chad (2002). . Penguin. .  \n'
 "The Complete Idiot's Guide to Taoism  \n"
 'ISBN  \n'
 '978-0-02-864262-8  \n'
 'Wachowski, Larry; Wachowski, Andy (2000). . Titan. .  \n'
 'The Ar

In [24]:
len(chunks)

128

In [11]:
chunks[0].page_content

'1999 film by the Wachowskis  \nThis article is about the 1999 film. For the franchise it initiated, see . For other uses, see .  \n(franchise)  \nThe Matrix  \nMatrix  \nThe Matrix  \nTheatrical release poster  \nDirected by  \nThe Wachowskis  \na  \n[  \n]  \nWritten by  \nThe Wachowskis  \nProduced by  \nJoel Silver  \nStarring  \nKeanu Reeves  \nLaurence Fishburne  \nCarrie-Anne Moss  \nHugo Weaving  \nJoe Pantoliano  \nCinematography  \nBill Pope  \nEdited by  \nZach Staenberg  \nMusic by  \nDon Davis  \nProduction companies  \nVillage Roadshow Pictures  \nGroucho II Film Partnership  \nSilver Pictures'

In [25]:
type(chunks)

list

In [12]:
cosine_collection.query(
    query_texts=["What year was 2001: A Space Odyssey released, and what genre is it?"],
    n_results=15
)

{'ids': [['html-2001_A_Space_Odyssey.html-149',
   'html-2001_A_Space_Odyssey.html-6',
   'html-2001_A_Space_Odyssey.html-88',
   'html-2001_A_Space_Odyssey.html-260',
   'html-2001_A_Space_Odyssey.html-205',
   'html-2001_A_Space_Odyssey.html-293',
   'html-2001_A_Space_Odyssey.html-7',
   'html-2001_A_Space_Odyssey.html-296',
   'html-2001_A_Space_Odyssey.html-156',
   'html-2001_A_Space_Odyssey.html-57',
   'html-2001_A_Space_Odyssey.html-255',
   'html-2001_A_Space_Odyssey.html-94',
   'html-2001_A_Space_Odyssey.html-0',
   'html-2001_A_Space_Odyssey.html-295',
   'html-2001_A_Space_Odyssey.html-208']],
 'embeddings': None,
 'documents': [['Section: Soundtrack\n\nMain article: 2001: A Space Odyssey (soundtrack)',
   "2001: A Space Odyssey Uptown Theater Metro-Goldwyn-Mayer 2001: A Space Odyssey human evolution technology artificial intelligence extraterrestrial life Academy Awards visual effects 5 [ ] The film is widely regarded as one of the . In 1991, it was selected by the Unite

In [28]:
cosine_collection_large_chunk.query(
    query_texts=["What year was 2001: A Space Odyssey released, and what genre is it?"],
    n_results=15
)

{'ids': [['html-2001_A_Space_Odyssey.html-2',
   'html-2001_A_Space_Odyssey.html-71',
   'html-2001_A_Space_Odyssey.html-0',
   'html-2001_A_Space_Odyssey.html-126',
   'html-2001_A_Space_Odyssey.html-3',
   'html-2001_A_Space_Odyssey.html-74',
   'html-2001_A_Space_Odyssey.html-27',
   'html-2001_A_Space_Odyssey.html-123',
   'html-2001_A_Space_Odyssey.html-43',
   'html-2001_A_Space_Odyssey.html-99',
   'html-2001_A_Space_Odyssey.html-1',
   'html-2001_A_Space_Odyssey.html-102',
   'html-2001_A_Space_Odyssey.html-136',
   'html-2001_A_Space_Odyssey.html-40',
   'html-2001_A_Space_Odyssey.html-98']],
 'embeddings': None,
 'documents': [["spaceflight special effects 2001: A Space Odyssey classical music its soundtrack Richard Strauss Johann Strauss II Aram Khachaturian György Ligeti premiered at on 2 April 1968 before being released by on 3 April 1968 in the United States and 1 May 1968 in the United Kingdom. Polarising critics after its release, has since been subject to a variety of 

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

In [30]:
cosine_collection_large_chunk.query(
    query_texts=["What year was 2001: A Space Odyssey released?", "What genre is 2001: A Space Odyssey?"],
    n_results=15
)

{'ids': [['html-2001_A_Space_Odyssey.html-2',
   'html-2001_A_Space_Odyssey.html-74',
   'html-2001_A_Space_Odyssey.html-0',
   'html-2001_A_Space_Odyssey.html-136',
   'html-2001_A_Space_Odyssey.html-3',
   'html-2001_A_Space_Odyssey.html-43',
   'html-2001_A_Space_Odyssey.html-1',
   'html-2001_A_Space_Odyssey.html-71',
   'html-2001_A_Space_Odyssey.html-126',
   'html-2001_A_Space_Odyssey.html-27',
   'html-2001_A_Space_Odyssey.html-99',
   'html-2001_A_Space_Odyssey.html-102',
   'html-2001_A_Space_Odyssey.html-123',
   'html-2001_A_Space_Odyssey.html-23',
   'html-2001_A_Space_Odyssey.html-98'],
  ['html-2001_A_Space_Odyssey.html-27',
   'html-2001_A_Space_Odyssey.html-123',
   'html-2001_A_Space_Odyssey.html-2',
   'html-2001_A_Space_Odyssey.html-126',
   'html-2001_A_Space_Odyssey.html-43',
   'html-2001_A_Space_Odyssey.html-71',
   'html-2001_A_Space_Odyssey.html-0',
   'html-2001_A_Space_Odyssey.html-3',
   'html-2001_A_Space_Odyssey.html-1',
   'html-2001_A_Space_Odyssey.html

In [31]:
cosine_collection_large_chunk.query(
    query_texts=["Who is the director of 2001: A Space Odyssey?"],
    n_results=10
)

{'ids': [['html-2001_A_Space_Odyssey.html-2',
   'html-2001_A_Space_Odyssey.html-0',
   'html-2001_A_Space_Odyssey.html-1',
   'html-2001_A_Space_Odyssey.html-126',
   'html-2001_A_Space_Odyssey.html-99',
   'html-2001_A_Space_Odyssey.html-27',
   'html-2001_A_Space_Odyssey.html-43',
   'html-2001_A_Space_Odyssey.html-3',
   'html-2001_A_Space_Odyssey.html-102',
   'html-2001_A_Space_Odyssey.html-98']],
 'embeddings': None,
 'documents': [["spaceflight special effects 2001: A Space Odyssey classical music its soundtrack Richard Strauss Johann Strauss II Aram Khachaturian György Ligeti premiered at on 2 April 1968 before being released by on 3 April 1968 in the United States and 1 May 1968 in the United Kingdom. Polarising critics after its release, has since been subject to a variety of interpretations, ranging from the darkly apocalyptic to an optimistic reappraisal of the hopes of humanity. Critics noted its exploration of themes such as , , , and the possibility of . It was nominate

In [14]:
collection_persistent.query(
    query_texts=["What year was 2001: A Space Odyssey released, and what genre is it?"],
)

{'ids': [['html-2001_A_Space_Odyssey.html-164',
   'html-2001_A_Space_Odyssey.html-433',
   'html-2001_A_Space_Odyssey.html-99',
   'html-2001_A_Space_Odyssey.html-375',
   'html-2001_A_Space_Odyssey.html-223',
   'html-2001_A_Space_Odyssey.html-434',
   'html-2001_A_Space_Odyssey.html-220',
   'html-2001_A_Space_Odyssey.html-10',
   'html-2001_A_Space_Odyssey.html-171',
   'html-2001_A_Space_Odyssey.html-395']],
 'embeddings': None,
 'documents': [['Main article: 2001: A Space Odyssey (soundtrack)',
   "v t e Space Odyssey Films (1968) 2001: A Space Odyssey (1984) 2010: The Year We Make Contact Novels (1968) 2001: A Space Odyssey (1982) 2010: Odyssey Two (1987) 2061: Odyssey Three (1997) 3001: The Final Odyssey Non-fiction The Lost Worlds of 2001 Comics 2001: A Space Odyssey Characters HAL 9000 Elements Monoliths Discovery Related Interpretations of 2001: A Space Odyssey Technologies in 2001: A Space Odyssey in popular culture 2001: A Space Odyssey soundtrack 2001: A Space Odyssey Ale